In [1]:
import carla

In [2]:
CARLA_HOST ="localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT = 10.0

TRAFFIC_MANAGER_PORT = 8000

In [3]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:
    '''Create a CARLA client and return the active world.'''

    client = carla.Client(
        CARLA_HOST,
        CARLA_PORT,
    )

    client.set_timeout(CARLA_TIMEOUT)
    world = client.get_world()

    return client, world

In [4]:
def get_vehicle_blueprint(world: carla.World) -> carla.ActorBlueprint:
    '''Select a vehicle blueprint.'''

    vehicle_blueprint = world.get_blueprint_library().filter("vehicle.*")

    if not vehicle_blueprint:
        raise RuntimeError(
            "Carla was not able to find specified actor blueprint."
        )

    return vehicle_blueprint[0]


In [5]:
def get_spawn_point(world: carla.World) -> carla.Transform:
    '''Return a valid spawn point.'''

    spawn_points = world.get_map().get_spawn_points()

    if not spawn_points:
        raise RuntimeError(
            '''Carla was not able to return a valid spawn point.'''
        )

    return spawn_points[0]

In [6]:
def spawn_vehicle(world: carla.World) -> carla.Vehicle:
    '''Spawn a vehicle at a valid spawn point.'''

    vehicle_bp = get_vehicle_blueprint(world)
    spawn_point = get_spawn_point(world)
    vehicle = world.try_spawn_actor(
        vehicle_bp,
        spawn_point,
    )

    if not vehicle:
        raise RuntimeError(
            "Vehicle was not able to be added into the world."
        )

    return vehicle


In [7]:
def set_autopilot(vehicle: carla.Vehicle, traffic_manager_port: int) -> None:
    '''Set autopilot.'''

    vehicle.set_autopilot(
        True,
        traffic_manager_port,
    )
    

In [8]:
def main() -> None:
    '''Spawn a vehicle and enable CARLA autopilot.'''

    client, world = connect_to_carla()

    vehicle = spawn_vehicle(
        world
    )

    try:
        set_autopilot(
            vehicle,
            TRAFFIC_MANAGER_PORT,
        )

        print("=== AUTOPILOT ENABLED ===")
        print(f"Actor ID: {vehicle.id}")
        print(f"Vehicle type: {vehicle.type_id}")
        print(
            f"Traffic Manager port: "
            f"{TRAFFIC_MANAGER_PORT}"
        )

        print()
        print(
            "Vehicle is now controlled by "
            "the CARLA Traffic Manager."
        )

        # Keep the script alive while the vehicle drives.
        input(
            "\nPress ENTER to stop the simulation..."
        )
    finally:
        if vehicle.is_alive:
            vehicle.set_autopilot(False)
            vehicle.destroy()

            print("Vehicle destroyed.")


In [9]:
if __name__ == "__main__":
    main()

=== AUTOPILOT ENABLED ===
Actor ID: 34
Vehicle type: vehicle.audi.a2
Traffic Manager port: 8000

Vehicle is now controlled by the CARLA Traffic Manager.
Vehicle destroyed.
